In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import torch
import os

from src.metric import *

In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer, util
from sentence_transformers.cross_encoder import CrossEncoder

from sklearn.metrics import ndcg_score

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
test_df=pd.read_csv('../data/raw/test.csv')

In [ ]:
bi_encoder=SentenceTransformer("../models/bi_encoder")

cross_encoder=CrossEncoder("../models/cross_encoder")

In [ ]:
bi_encoder_hn=SentenceTransformer("../models/bi_encoder_hard_negative")

cross_encoder_hn = CrossEncoder("../models/cross_encoder_hard_negative")

In [ ]:
test_resume_texts=test_df['resume_text'].drop_duplicates().tolist()

test_resume_embs=bi_encoder.encode(test_resume_texts,convert_to_tensor=True,
                             batch_size=64,show_progress_bar=True)


In [ ]:
def retrieve_candidates(jd,label_map,model,test_resume_embs,test_resume_texts,top_k=50):

    jd_emb=model.encode(jd,convert_to_tensor=True)

    hits=util.semantic_search(jd_emb,test_resume_embs,top_k=top_k)
    hits=hits[0]

    retrieval_rows=[]

    for hit in hits:

        idx=hit["corpus_id"]

        resume_text=test_resume_texts[idx]

        if resume_text not in label_map:
            continue

        retrieval_rows.append({
            "resume_text": resume_text,
            "score": float(hit["score"]),
            "label": label_map[resume_text]
        })

    return pd.DataFrame(retrieval_rows)

In [ ]:
def rerank_candidates(jd,retrieval_df,model):

    pairs=list(zip([jd]*len(retrieval_df),retrieval_df['resume_text']))

    cross_scores=model.predict(pairs,batch_size=64)

    rerank_df=retrieval_df.copy()

    rerank_df["score"]=cross_scores

    rerank_df=rerank_df.sort_values("score",ascending=False).reset_index(drop=True)

    return rerank_df

In [ ]:
def evaluate_df(df):
    metrics = {}

    metrics["ndcg"] = ndcg_metric(df)

    metrics["spearman"] = corr_metric(df)

    metrics["topk"] = topk_metric(df)

    metrics["mrr"] = mrr_metric(df)

    metrics["map"] = map_metric(df)

    return metrics

In [ ]:
bi_encoder_metrics={
    "ndcg": [],
    "spearman": [],
    "topk": [],
    "map": [],
    "mrr": []
}
cross_encoder_metrics={
    "ndcg": [],
    "spearman": [],
    "topk": [],
    "map": [],
    "mrr": []
}

In [ ]:
for jd, group in test_df.groupby("job_description_text"):

    if len(group)<2:
        continue
        
    group_label=group[['resume_text','label']].drop_duplicates()
    label_map=dict(zip(group_label['resume_text'],group_label['label']))

    # Stage 1 Retrieval

    retrieval_df=retrieve_candidates(jd,label_map,bi_encoder,
                                       test_resume_embs,test_resume_texts,
                                       top_k=50)

    if retrieval_df.empty:
        continue

    bi_metrics=evaluate_df(retrieval_df)
    
    for key in bi_encoder_metrics:
        if bi_metrics[key] is not None:
            bi_encoder_metrics[key].append(bi_metrics[key])

    # Stage 2 Reranking

    rerank_df=rerank_candidates(jd,retrieval_df,cross_encoder)

    cross_metrics=evaluate_df(rerank_df)
    
    for key in cross_encoder_metrics:
        if cross_metrics[key] is not None:
            cross_encoder_metrics[key].append(cross_metrics[key])

In [ ]:
results_df=pd.DataFrame({
    "metric": list(bi_encoder_metrics.keys()),
    "bi_encoder": [np.mean(vals) if vals else 0 for vals in bi_encoder_metrics.values()],
    "cross_encoder": [np.mean(vals) if vals else 0 for vals in cross_encoder_metrics.values()]
})

In [ ]:
print("\n" + "="*60)
print("FINAL TWO-STAGE RETRIEVAL RESULTS")
print("="*60)

for _, row in results_df.iterrows():
    print(
        f"{row['metric']:<20}"
        f"{row['bi_encoder']:>10.4f}"
        f"{row['cross_encoder']:>15.4f}"
    )

print("="*60)

In [ ]:
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/two_stage_ranking_results.csv',index=False)